# Faz 3 v3 — HFP Grafting: Qwen2.5-1.5B + O(1) Bellek (Distilasyon)

v2'nin temiz yeniden kurulumu. **Yenilikler (v2 kosusunun dersleri):**
- **Kesinti sigortasi:** Google Drive baglanirsa checkpoint'ler oraya yazilir (oturum dusse de kalir); her 250 adimda kayit.
- **RESUME:** onceki kosudan inen `hfp_graft_stage1_*.pt` / `hfp_graft_stage2_*.pt` dosyasiyla kaldigi yerden devam (Hucre 4).
- **S1 dersi:** ilk kosuda MSE ~1000-1150 adimda 0.07 platosuna oturdu -> `STEPS1=1200` (gerekirse artir).
- GPU assert + NaN korumalari.

**Kaggle iki-versiyon plani (12h siniri yuzunden):**
- **Versiyon 1:** Save & Run All → 1-5 kosar (S1, ~10h), S2/validasyon otomatik atlanir → versiyon TAMAMLANIR, `hfp_graft_stage1_son.pt` Output'ta kalir.
- **Versiyon 2:** Add Input → kendi notebook'unun v1 ciktisini ekle → Save & Run All → Hucre 4 resume eder, S1 atlanir, S2 (~8.5h) + validasyon kosar → `hfp_graft_final.pt`.
- Ayni oturumda S1+S2 zorlamak icin (Colab/uzun kota): Hucre 6'dan once `FORCE_S2 = True`.

**Sure tahmini (T4):** S1 ~0.83 dk/adim (1200 adim ≈ 16-17 sa; resume ile 0), S2 ilk 50-adim logunda dk/adim gorulur.
**On-kayitli kriterler (SONRAKI_ADIMLAR_PLANI.md K3):** (a) zero-shot PPL<1000, (b) S2 sonrasi PPL ≤ 1.05× orijinal (1.05-1.20 kabul edilebilir ilk gecis), (c) alpha dagilimi kaydi, (d) needle 2K sanity, (e) O(1) state raporu.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys
import importlib

# [FIX] HF Xet backend'i Colab/Kaggle'da buyuk dosyalarda 0%'da stall ediyor
# (model.safetensors ?B/s). Klasik CDN indirmesine zorla.
# [FIX] T4'te Stage 2 OOM'a karsi: bellek parcalanmasini azalt (kil payi OOM'lari kurtarir).
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
# NOT: hf-mirror.com bu Kaggle'dan erisilemiyordu -> resmi HF + HF_TOKEN kullaniyoruz.
# Model resmi HF'den (token'li) iner; WikiText-103 ise xet-bridge 403'unu asmak icin
# HF datasets yerine dogrudan S3 raw indirmeyle gelir (Hucre 5).

# [SIGORTA] GPU assert — CPU'da 15 saat bosa kosma dersi
import torch
assert torch.cuda.is_available(), 'GPU SECILI DEGIL! Runtime > Change runtime type > T4 GPU.'
# [SIGORTA] Mimari uyumu: yeni torch derlemeleri P100/Pascal'i (sm_60) desteklemiyor
# ("no kernel image" hatasi). Kaggle'da P100 DEGIL, T4 / T4 x2 secin.
_cap = torch.cuda.get_device_capability(0)
_arch = f'sm_{_cap[0]}{_cap[1]}'
assert _arch in torch.cuda.get_arch_list(), (
    f'GPU mimarisi {_arch} bu torch derlemesinde YOK {torch.cuda.get_arch_list()} '
    f'-> Accelerator olarak T4 secin (P100 desteklenmiyor).')
DEV = 'cuda'

# [FIX] Kaggle imajinda /content klasoru DE var -> once /kaggle/working.
# Kaggle yalniz /kaggle/working'i versiyon ciktisi olarak saklar; /content'e
# yazilan checkpoint'ler oturumla birlikte YOK OLUR (yasandi).
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers>=4.46', 'datasets', 'accelerate'], check=True)
# ONERI: Model ve WikiText-103 icin en saglam yol Kaggle INPUT (sag panel > Add Input).
# Bu, HF'nin Xet CDN 403'unu tamamen atlar. HF indirme yalniz yedek (Hucre 2/5).
os.chdir(REPO); sys.path.insert(0, REPO)

# [SIGORTA] Checkpoint'ler icin Drive (varsa) — oturum dusse de dosyalar kalir
CKPT_DIR = BASE
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/hfp_graft_ckpt'
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('Checkpoint dizini (Drive):', CKPT_DIR)
except Exception as e:
    print(f'Drive yok/reddedildi ({type(e).__name__}) -> checkpoint {BASE} altina yazilir; ARA ARA INDIR!')

# HF girisi: Qwen2.5-1.5B UNGATED -> token OPSIYONEL. Prompt YOK (arka plan
# kosumlarinda bloklar); varsa Colab/Kaggle Secrets'tan alinir.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
    print('HF token Colab Secrets\'tan alindi.')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
        print('HF token Kaggle Secrets\'tan alindi.')
    except Exception:
        print('HF token yok — Qwen2.5-1.5B ungated, tokensiz devam.')

print(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0)}')

# graft_smoke: torch ortaminda ILK kosum — gecmeden devam etmeyin
env = dict(os.environ, PYTHONPATH=REPO + ':' + os.environ.get('PYTHONPATH', ''))
r = subprocess.run([sys.executable, 'review_scripts/graft_smoke.py'],
                   capture_output=True, text=True, env=env, cwd=REPO)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'graft_smoke.py BASARISIZ — devam etmeyin'


In [ ]:
# --- 2. MODEL + GRAFT ONCESI REFERANS PPL ---
# [FIX v6] HF Xet CDN (xet-bridge) bu Kaggle bolgesinde model.safetensors icin
# TOKEN'LA BILE 403 veriyor; hf-mirror erisimsiz. EN SAGLAM: modeli Kaggle INPUT
# olarak ekle (harici indirme yok). Once Input'ta Qwen aranir; yoksa HF denenir.
import math, os, glob, urllib.request, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B'

def _find_local_model():
    # Kaggle Input VE Colab (Drive/content) icinde config.json + *.safetensors ara.
    roots = ['/kaggle/input', globals().get('CKPT_DIR'), BASE,
             '/content/drive/MyDrive', '/content']
    seen = set()
    for root in roots:
        if not root or root in seen or not os.path.isdir(root):
            continue
        seen.add(root)
        for cfg in glob.glob(f'{root}/**/config.json', recursive=True):
            d = os.path.dirname(cfg)
            if glob.glob(f'{d}/*.safetensors'):
                return d
    return None

LOCAL_MODEL = _find_local_model()
if LOCAL_MODEL:
    print(f'Model KAGGLE INPUT tan yuklendi (indirme yok): {LOCAL_MODEL}')
else:
    print('Kaggle Input ta Qwen yok -> HF den indirme denenecek (xet 403 riski).', flush=True)
    from huggingface_hub import snapshot_download
    _tk = os.environ.get('HF_TOKEN') or None
    assert _tk, (
        'Model bulunamadi ve HF_TOKEN yok.\n'
        '  KAGGLE: sag panel > Add Input > "Qwen2.5-1.5B" (Models) ekle.\n'
        '  COLAB: ya anahtar (Secrets) ikonundan HF_TOKEN ekle, ya da Qwen klasorunu\n'
        '  Drive: MyDrive/ altina yukle (kod otomatik bulur).')
    LOCAL_MODEL = f'{BASE}/qwen_model'
    snapshot_download(repo_id=MODEL_ID, local_dir=LOCAL_MODEL, token=_tk,
                      allow_patterns=['*.json', '*.txt', 'tokenizer.model',
                                      '*.safetensors', '*.safetensors.index.json'],
                      max_workers=2)
    _sft = [f for f in os.listdir(LOCAL_MODEL) if f.endswith('.safetensors')]
    assert _sft and all(os.path.getsize(f'{LOCAL_MODEL}/{f}') > 1_000_000 for f in _sft), (
        'HF indirme basarisiz (muhtemel Xet 403).\n'
        '  KAGGLE: Add Input ile Qwen2.5-1.5B ekle. COLAB: Qwen klasorunu Drive/MyDrive\n'
        '  altina yukle; kod otomatik bulur.')

tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=torch.float32).to(DEV)
model.eval()
print(f'{MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B param')

# WikiText-2 valid (PPL referansi) — github raw, xet YOK.
WT2 = 'https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/'
for fn in ('train.txt', 'valid.txt'):
    dst = f'{BASE}/wt2_{fn}'
    if not os.path.exists(dst):
        print(f'  indiriliyor: {fn}', flush=True)
        urllib.request.urlretrieve(WT2 + fn, dst)
train_text = open(f'{BASE}/wt2_train.txt', encoding='utf-8').read()
val_text = open(f'{BASE}/wt2_valid.txt', encoding='utf-8').read()
val_ids = tok(val_text, return_tensors='pt').input_ids[0]
print(f'Val: {len(val_ids):,} token | Train metni: {len(train_text):,} karakter')

@torch.no_grad()
def ppl(model, ids, seq=1024, n_chunks=24):
    model.eval(); losses = []
    for i in range(n_chunks):
        s = i * seq
        if s + seq > len(ids): break
        x = ids[s:s+seq].unsqueeze(0).to(DEV)
        out = model(x, labels=x)
        losses.append(out.loss.item())
    return math.exp(sum(losses)/len(losses))

PPL_BASE = ppl(model, val_ids)
print(f'GRAFT ONCESI referans PPL (WT-2 valid): {PPL_BASE:.2f}')


In [ ]:
# --- 3. GRAFT + SIFIRINCI-ADIM SANITY ---
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode, set_distill_backward,
                                 distill_loss, trainable_parameters,
                                 enable_streaming, reset_streaming, HFPGraftAttention)

SAVE_KEYS = ('decay','log_eta','conv_q','conv_k','beta_gate','alpha_logit','retrieval_norm','out_gain')
def save_hfp(tag):
    _rt = f"{DECAY_MODE}_g{len(GRAFT_LAYERS)}{SEL}_s{RUN_SEED}"
    path = f'{CKPT_DIR}/hfp_graft_{_rt}_{tag}.pt' 
    torch.save({n: p for n, p in model.state_dict().items()
                if any(k in n for k in SAVE_KEYS)}, path)
    print(f'  checkpoint: {path}', flush=True)

# [RUN6 — KONTROLLU KIYAS] Tek degisken: retention yasasi.
#   'cubic_flux_chunked' = Run 5 (referans)  |  'exp' = ikiz kosu
# Diger HER SEY (S1/S2 protokolu, init, veri, seed) birebir ayni kalir.
# ===================== KOSU PARAMETRELERI (tek yer) =====================
# [FAZ-B — §24] Ilkeli-13 secim testi: GRAFT_FROM_MAP=13 (guarded-cheapest-13) -> PPL vs odd-13 (1.6x).
# Cok-seed replikasyon TAMAMLANDI (Run 7/8/9, 3/3; §22b/c). Bu kosu FAZ-B; seed 0 (baseline-esli).
RUN_SEED = 0
GRAFT_N  = 6           # referans recete: seyrek graft [3,7,11,15,19,23] (§22a)
GRAFT_FROM_MAP  = 13    # [FAZ-B] int K -> layer_linearization_map.json'dan EN-UCUZ-K sec (Faz-A ciktisi). None=normal
GRAFT_MAP_GUARD = True  # [FAZ-B/§24 H2] ilk(0)+son(N-1) katmani havuzdan CIKAR: NMSE onlari
                        #   yaniltici ucuz gosteriyor ama LM-head'e dogrudan bagli -> PPL-kritik.
SEL = ''                # checkpoint etiket eki; map secimi 'map'/'mapg' yapar (cakismayi onler)
FORCE_S2 = True        # tek oturumda S1+S2 (S1 seq128'de ~1h, sigar)
S2_STAB  = True        # [§18] 13-kat egitim STABILIZASYON probu: dusuk LR + uzun warmup + uzun S2
                       #   (Faz-B S2 loss'u 30-155 salindi -> trainability mi kapasite mi? testi)
S1_STUDENT_FORWARD = True   # [§25] Stage-1 STUDENT-FORWARD distilasyon: exposure-bias/compounding KOK cozumu
SF_WARMUP_FRAC     = 0.4    #   ilk %40 teacher-forcing (temiz isinma) -> sonra student-forcing (gercek bozuk girdi)
import random as _r
_r.seed(RUN_SEED); torch.manual_seed(RUN_SEED)   # graft init + veri sirasi tekrarlanabilir
# ========================================================================
DECAY_MODE = 'exp'
gcfg = GraftConfig(decay_mode=DECAY_MODE, write_rule='hybrid',
                   key_feature_map='dpfp', rec_block=16)   # [VRAM] T4: kucuk blok sart
if GRAFT_FROM_MAP:
    # [FAZ-B] Faz-A haritasindan EN-UCUZ-K katmani sec (ilkeli secim; tek degisken = hangi katmanlar).
    import glob as _g, json as _j
    _mapf = None
    for _d in [globals().get('CKPT_DIR'), '/content/drive/MyDrive/hfp_bench',
               f'{REPO}/docs/assets', BASE, '/kaggle/input']:
        if not _d: continue
        _hits = _g.glob(f'{_d}/**/layer_linearization_map.json', recursive=True)
        if _hits: _mapf = _hits[0]; break
    assert _mapf, ('layer_linearization_map.json bulunamadi. Once Faz-A '
                   '(layer_linearization_probe) kosup JSON uret (Drive/hfp_bench veya docs/assets). '
                   'Kaggle: Add Input ile ekle.')
    _nm = {int(k): v for k, v in _j.load(open(_mapf))['nmse'].items()}
    _pool = list(_nm)
    if GRAFT_MAP_GUARD:                                  # §24 H2: sinir katmanlarini koru
        _pool = [i for i in _pool if i not in (0, model.config.num_hidden_layers - 1)]
    _layers = sorted(sorted(_pool, key=lambda i: _nm[i])[:GRAFT_FROM_MAP])
    SEL = 'mapg' if GRAFT_MAP_GUARD else 'map'
    if S2_STAB: SEL += 'S'          # [§18] stabilize kolu -> ayri checkpoint soyu (cakisma yok)
    if S1_STUDENT_FORWARD: SEL += 'F'   # [§25] student-forward kolu -> ayri soy
    _odd = [i for i in range(1, model.config.num_hidden_layers-1) if i % 2 == 1]
    print(f'[FAZ-B] harita: {_mapf}  (sinir-koruma={GRAFT_MAP_GUARD})')
    print(f'[FAZ-B] ilkeli-{GRAFT_FROM_MAP} secim: {_layers}')
    print(f'[FAZ-B] kiyas tabani: §22a odd-13 = {_odd} (PPL 1.6x). Kriter (§24): belirgin < 1.6x -> secim ise yariyor.')
else:
    _layers = None if GRAFT_N == 13 else [3, 7, 11, 15, 19, 23]   # 6'li: esit araliklı
GRAFT_LAYERS = graft_llama(model, gcfg, layers=_layers)
# [RUN2 — tek degisken] out_gain init 1.0 -> 0.1. Gerekce: teshis T2
# (kaggle_graft_diagnostics_v1): zero-shot PPL 2758(1.0) / 168(0.1) / 4705(0.01).
# Kotu baslangic noktasi hipotezi (RESULTS §15). Baska HICBIR sey degismedi.
for _m in model.modules():
    if isinstance(_m, HFPGraftAttention): _m.out_gain.data.fill_(0.1)
print('[RUN2] out_gain init = 0.1 (T2 teshisine gore)')

# teacher modu == orijinal model (dogrulama) — AYNI chunk sayisiyla kiyasla!
set_graft_mode(model, 'teacher')
ppl_teacher = ppl(model, val_ids)            # PPL_BASE ile ayni 24 chunk
print(f'teacher {ppl_teacher:.2f} vs base {PPL_BASE:.2f}')
assert abs(ppl_teacher - PPL_BASE) < PPL_BASE*0.01, 'teacher yolu bozuk!'

# student modu, egitimsiz: PPL kotu olacak ama COP OLMAMALI (kriter: <1000)
set_graft_mode(model, 'student')
PPL_ZERO = ppl(model, val_ids, n_chunks=8)
print(f'Zero-shot (egitimsiz HFP) PPL: {PPL_ZERO:.1f}  (beklenti ~168, kriter <1000: {"GECTI" if PPL_ZERO < 1000 else "KALDI — agirlik transferi/olcek kontrol"})')


In [ ]:
# --- 4. RESUME (OPSIYONEL): onceki kosunun checkpoint'inden devam ---
# Kullanim: .pt dosyasini sol Files paneline (veya Drive'daki hfp_graft_ckpt/)
# surukle; RESUME_PATH bos birakilirsa en yenisi otomatik bulunur.
import glob, torch

RESUME_PATH = ''          # orn: '/content/hfp_graft_stage1_kesildi_1150.pt' — bos = otomatik ara
SKIP_S1 = False           # resume stage1>=1000 ise asagida otomatik True yapilir

if not RESUME_PATH:
    # [FIX] recursive: input alt-klasorlerde de bulur; 'son' varsa onu tercih et.
    _all = (glob.glob(f'{CKPT_DIR}/**/hfp_graft_*.pt', recursive=True)
            + glob.glob(f'{BASE}/**/hfp_graft_*.pt', recursive=True)
            + glob.glob('/kaggle/input/**/hfp_graft_*.pt', recursive=True))
    _son = [x for x in _all if 'son' in os.path.basename(x)]
    cands = sorted(_son or _all, key=os.path.getmtime)
    RESUME_PATH = cands[-1] if cands else ''

# [RUN8/9 — SEED KORUMASI] Drive kalici oldugundan farkli seed/yogunluk
# kosusunun checkpoint'i bulunabilir; yalniz BU kosunun etiketi kabul edilir.
_rt_now = f"{DECAY_MODE}_g{len(GRAFT_LAYERS)}{SEL}_s{RUN_SEED}"
if RESUME_PATH and _rt_now not in os.path.basename(RESUME_PATH):
    print(f'UYARI: {os.path.basename(RESUME_PATH)} bu kosunun etiketi ({_rt_now}) degil -> resume IPTAL, sifirdan S1.')
    RESUME_PATH = ''
# [RUN6] MOD KORUMASI: exp kosusuna cubic checkpoint'i (veya tersi) yuklenmez —
# parametre adlari ayni oldugu icin sessizce yuklenir ama SEMANTIK yanlis olur.
if RESUME_PATH:
    _is_exp_ckpt = 'exp_' in os.path.basename(RESUME_PATH)
    _want_exp = (DECAY_MODE == 'exp')
    if _is_exp_ckpt != _want_exp:
        print(f'UYARI: {os.path.basename(RESUME_PATH)} mod uyusmuyor (DECAY_MODE={DECAY_MODE}) -> resume IPTAL, sifirdan S1.')
        RESUME_PATH = ''
if RESUME_PATH:
    sd = torch.load(RESUME_PATH, map_location=DEV)
    res = model.load_state_dict(sd, strict=False)
    n_loaded = len(sd) - len(res.unexpected_keys)
    assert n_loaded > 0 and not res.unexpected_keys, f'checkpoint uyumsuz: {res.unexpected_keys[:5]}'
    alphas = [torch.sigmoid(m.alpha_logit).mean().item()
              for m in model.modules() if isinstance(m, HFPGraftAttention)]
    print(f'RESUME: {RESUME_PATH}  ({n_loaded} tensor yuklendi)  alpha_ort {sum(alphas)/len(alphas):.3f}')
    if 'stage1' in RESUME_PATH:
        import re
        m = re.search(r'(\d+)', RESUME_PATH.split('stage1')[-1])
        if (m and int(m.group(1)) >= 700) or 'son' in RESUME_PATH:   # plato ~700-800 (iki kosu kaniti)
            SKIP_S1 = True
            print('-> S1 yeterince kosulmus: SKIP_S1=True (Hucre 5 atlanacak, dogrudan Hucre 6/S2).')
    if 'stage2' in RESUME_PATH or 'final' in RESUME_PATH:
        SKIP_S1 = True
        print('-> S2 checkpoint: Hucre 5 atlanir; S2 kalan adimlar icin STEPS2 dusurulebilir.')
else:
    print('Checkpoint bulunamadi — sifirdan S1 (Hucre 5) kosulacak.')


In [ ]:
# --- 5. STAGE 1: TEACHER-FORCING DISTILASYON (katman-ici MSE) ---
# Ileriye ogretmen ciktisi gider -> katmanlar bagimsiz, akis on-distribution.
# [v3] Ilk kosu dersi: MSE ~1000-1150 adimda 0.07 platosuna oturdu -> STEPS1=700.
import time, os, glob, zipfile, urllib.request
from transformers import get_cosine_schedule_with_warmup

SEQ, BATCH, STEPS1, LR1 = 128, 1, 700, 1e-3   # [T4-OOM] Stage 2 icin seq 1024->512 (float32 korunur; bellek ~yariya)
ACCUM = 4

# WT-103 bulunamazsa: WT-2'ye SESSIZCE dusme YOK. Bilerek WT-2 istiyorsan True yap.
ALLOW_WT2_FALLBACK = False

# [FIX v6] Egitim korpusu secimi (HF datasets/xet KULLANMADAN):
#   1) WT-103 Kaggle Input (en iyi)  2) WT-103 S3 raw indirme
#   3) WT-2 YEDEK (Hucre 2'de zaten indi) — kucuk korpus, DENEY SAPMASI olarak isaretlenir.
def _find_local_wt103():
    for pat in ('wiki.train.raw', 'wiki.train.tokens', '*train*.raw', '*train*.tokens'):
        hits = glob.glob(f'/kaggle/input/**/{pat}', recursive=True)
        if hits: return hits[0]
    return None

TRAIN_RAW = _find_local_wt103()
_CORPUS = 'WT-103 (Kaggle Input)'
if TRAIN_RAW is None:
    WT103_DIR = f'{BASE}/wikitext-103-raw'
    _wt103 = f'{WT103_DIR}/wiki.train.raw'
    _ZIP_URLS = [
        'https://wikitext.smerity.com/wikitext-103-raw-v1.zip',
        'https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-103-raw-v1.zip',
        'https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-103-raw-v1.zip',
    ]
    def _fetch(url, dest):
        # UA + redirect izleme (301/403 sebeplerini asar); urlretrieve bunlari yapmiyordu.
        import urllib.request as _u
        req = _u.Request(url, headers={'User-Agent': 'Mozilla/5.0', 'Accept': '*/*'})
        with _u.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
            while True:
                b = r.read(1 << 20)
                if not b: break
                f.write(b)
    try:
        if not os.path.exists(_wt103):
            zip_path = f'{BASE}/wt103-raw.zip'
            if not (os.path.exists(zip_path) and os.path.getsize(zip_path) > 1_000_000):
                _ok = False
                for u in _ZIP_URLS:
                    try:
                        print(f'WT-103 raw indiriliyor (~190MB): {u}', flush=True)
                        _fetch(u, zip_path)
                        if os.path.getsize(zip_path) > 1_000_000:
                            _ok = True; break
                    except Exception as e:
                        print(f'  basarisiz: {type(e).__name__}: {str(e)[:100]}')
                assert _ok, 'WT-103 zip inmedi'
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(BASE)
        assert os.path.exists(_wt103)
        TRAIN_RAW = _wt103; _CORPUS = 'WT-103 (S3 raw)'
    except Exception as e:
        if not ALLOW_WT2_FALLBACK:
            raise RuntimeError(
                'WT-103 alinamadi (' + type(e).__name__ + ': ' + str(e)[:120] + ').\n'
                '  S3/mirror bu ortamda engelli olabilir. COZUM: sag panel > Add Input >\n'
                '  Datasets sekmesinde "wikitext-103" (veya "wikitext103") ara ve icinde\n'
                '  wiki.train.raw / wiki.train.tokens olan bir dataseti ekle; kod otomatik bulur.\n'
                '  (Bilerek kucuk WT-2 ile denemek istersen hucre basindaki\n'
                '   ALLOW_WT2_FALLBACK = True yap — bu bir DENEY SAPMASIDIR.)') from e
        TRAIN_RAW = f'{BASE}/wt2_train.txt'    # yalnizca ALLOW_WT2_FALLBACK=True ise
        _CORPUS = 'WT-2 (YEDEK — kucuk korpus, BILEREK secildi)'
        print('WT-103 yok -> WT-2 yedegi (ALLOW_WT2_FALLBACK=True).', flush=True)

assert os.path.exists(TRAIN_RAW), f'Egitim korpusu bulunamadi: {TRAIN_RAW}'
print(f'Egitim korpusu: {_CORPUS} -> {TRAIN_RAW} ({os.path.getsize(TRAIN_RAW)/1e6:.1f} MB)', flush=True)
if _CORPUS.startswith('WT-2'):
    print('  [UYARI/SAPMA] WT-103 yerine WT-2 kullaniliyor: daha kucuk ve az cesitli korpus.\n'
          '  Bu bir DENEY SAPMASIDIR -> RESULTS.md/rapor bunu ACIKCA belirtmeli. Asil kosu\n'
          '  icin WT-103 dataseti Add Input ile ekleyip yeniden kosun.', flush=True)

def batch_iter(path, seq, batch):
    buf = []
    while True:                       # veri biterse bastan sar (WT-2 kucuk oldugu icin gerekli)
        with open(path, encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                buf.extend(tok(line).input_ids)
                while len(buf) >= batch * (seq + 1):
                    chunk = buf[:batch*(seq+1)]; buf = buf[batch*(seq+1):]
                    t = torch.tensor(chunk).view(batch, seq+1)
                    yield t[:, :-1], t[:, 1:]

it = batch_iter(TRAIN_RAW, SEQ, BATCH)

if globals().get('SKIP_S1', False):
    print('SKIP_S1=True -> Stage 1 atlandi (resume edilen agirliklarla S2ye gecin).')
else:
    model.config.use_cache = False
    opt = torch.optim.AdamW(trainable_parameters(model), lr=LR1, weight_decay=0.01)
    sch = get_cosine_schedule_with_warmup(opt, 100, STEPS1)
    set_graft_mode(model, 'teacher_forcing')
    set_distill_backward(model, 1.0 / ACCUM)
    model.train()

    t0 = time.time(); log = []
    _sf_at = int(SF_WARMUP_FRAC * STEPS1) if globals().get('S1_STUDENT_FORWARD') else 10**9
    for step in range(1, STEPS1 + 1):
        if step == _sf_at + 1:                       # [§25] isinmadan sonra student-forward'a gec
            set_graft_mode(model, 'student_forcing')
            print(f'[§25] adim {step}: STUDENT-FORWARD distilasyona gecildi '
                  f'(her katman artik GERCEK bozuk girdi gorur; exposure-bias cozumu)', flush=True)
        loss_acc = 0.0
        for _ in range(ACCUM):
            x, _ = next(it)
            model(x.to(DEV))
            aux = distill_loss(model)
            loss_acc += aux.item() / ACCUM
        assert loss_acc == loss_acc, f'NaN @ S1 step {step}'
        torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        if step % 50 == 0 or step <= 3:
            alphas = [torch.sigmoid(m.alpha_logit).mean().item()
                      for m in model.modules() if isinstance(m, HFPGraftAttention)]
            print(f'S1 {step}/{STEPS1}  mse {loss_acc:.5f}  alpha_ort {sum(alphas)/len(alphas):.3f}  {(time.time()-t0)/60:.0f}dk', flush=True)
            log.append((step, loss_acc))
        if step % 250 == 0:
            save_hfp(f'stage1_{step}')
    save_hfp('stage1_son')
    print(f'Stage 1 tamam. (korpus: {_CORPUS})')


In [ ]:
# --- 6. STAGE 2: LOGIT-KL + LM LOSS (uctan uca) ---
# Ogretmen logitleri ayni modelin "teacher" modundan (ikinci kopya yok).
# [OOM-FIX] Onceki denemelerin artiklarini temizle + bellek durumunu raporla.
# NOT: OOM alan hucreyi restartsiz tekrar calistirmak, eski traceback'in tuttugu
# tensorler yuzunden bellegi BIRIKTIRIR. Temiz kosum: Restart -> 1..6 sirayla BIR kez.
import gc, sys, torch
sys.last_traceback = None   # OOM traceback'inin tuttugu tensorleri birak
gc.collect(); torch.cuda.empty_cache()
_alloc = torch.cuda.memory_allocated()/1e9
print(f'[bellek] Stage 2 oncesi allocated: {_alloc:.2f} GB')
assert _alloc < 8.5, (
    f'GPU zaten {_alloc:.1f} GB dolu (model ~6.2 GB olmali). Onceki kosum artigi var.\n'
    '  COZUM: Runtime/Session RESTART edin, sonra hucreleri 1->6 SIRAYLA ve BIRER KEZ kosun.')
import time
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup

S2_DONE = False
if not (globals().get('SKIP_S1', False) or globals().get('FORCE_S2', False)):
    print('Bu oturum S1 oturumu -> S2 atlandi. S2 icin: yeni versiyonda Add Input ile')
    print('onceki versiyonun ciktisini ekle -> Hucre 4 resume eder -> S2 kosar.')
    print('(Ayni oturumda S1+S2 zorlamak icin: FORCE_S2 = True)')
else:
    if globals().get('S2_STAB'):
        STEPS2, LR2, KL_W, TEMP, WARM2 = 900, 1e-4, 0.5, 2.0, 150   # [§18] stabilize
        print('[§18] S2 STABILIZASYON: LR 1e-4, warmup 150, 900 adim (13-kat trainability probu)')
    else:
        STEPS2, LR2, KL_W, TEMP, WARM2 = 600, 3e-4, 0.5, 2.0, 50
    # [RUN5 — tek degisken: MESAFE MUFREDATI] Cross-chunk recall + degisken ara.
    # RUN4 kaniti: @512 BULDU (bitisik A->B, ~3 chunk tasima) ama @2048 MISS ->
    # uzunluk-genellemesi darbogazi. DUZELTME: A ile B arasina 0-12 rastgele
    # dolgu chunk'i -> egitilen tasima mesafesi ~13 chunk (~1.7k token).
    # RUN3 kusuru: parola+sorgu ayni 128'lik penceredeydi -> dokunulmamis
    # tam-attention katmanlari pencere icinde cozuyordu; bellege baski yoktu.
    # DUZELTME: parola CHUNK A'da, sorgu CHUNK B'de. Egitimde cache yok ->
    # attention A'yi GOREMEZ; bilgi yalnizca recurrent state (M,z) ile tasinir.
    # Teacher KL hedefi tam-attention'la (A+B birlesik) uretilir.
    # SINIR (durust not): stream state chunk sinirinda detach'li -> cross-chunk
    # gradyan yalniz OKUMA yoluna akar; yazma yolu B-ici yazmalardan ogrenir.
    RECALL_MIX = 0.25
    import random as _rnd
    from hfp.models.grafting import enable_streaming, reset_streaming
    _RW = ['falcon','mountain','river','copper','silent','blue','amber','stone','harbor','willow']
    _fill_ids = tok(' The weather was ordinary and nothing of note happened that day.',
                    add_special_tokens=False).input_ids
    def _fill(n):
        b = []
        while len(b) < n: b.extend(_fill_ids)
        return b[:n]
    _GAPS = [0, 1, 2, 3, 4, 6, 8, 12]                     # mufredat: 0..12 ara chunk
    def _recall_crosschunk(seq):
        secret = _rnd.choice(_RW) + ' ' + _rnd.choice(_RW)
        n_ids = tok(f' The secret passphrase is {secret}.', add_special_tokens=False).input_ids
        q_ids = tok(f' The secret passphrase is {secret}', add_special_tokens=False).input_ids
        ba = _fill(seq - len(n_ids)); ins = _rnd.randrange(max(1, len(ba)))
        xa = (ba[:ins] + n_ids + ba[ins:])[:seq]          # CHUNK A: parola gomulu
        k = _rnd.choice(_GAPS)                            # ara mesafe (chunk sayisi)
        xg = torch.tensor(_fill(k * seq)).unsqueeze(0) if k else None
        xb = _fill(seq - len(q_ids)) + q_ids              # CHUNK B: sorgu+cevap sonda
        return (torch.tensor(xa).unsqueeze(0), xg, torch.tensor(xb).unsqueeze(0))
    _base_it = it
    def _mixed():
        while True:
            if _rnd.random() < RECALL_MIX:
                yield ('rc',) + _recall_crosschunk(SEQ)
            else:
                _x, _y = next(_base_it)
                yield ('lm', _x, None, None)
    it = _mixed()
    print(f'[RUN5] S2 verisi: WT-103 %{int((1-RECALL_MIX)*100)} + mesafe-mufredatli CROSS-CHUNK recall %{int(RECALL_MIX*100)}')
    # [VRAM] Qwen vocab 151k -> logit tensorleri buyuk; checkpointing T4 icin sart
    set_distill_backward(model, None)   # Stage 2'de kapali (uctan uca graf)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    opt = torch.optim.AdamW(trainable_parameters(model), lr=LR2, weight_decay=0.01)
    sch = get_cosine_schedule_with_warmup(opt, WARM2, STEPS2)

    t0 = time.time()
    for step in range(1, STEPS2 + 1):
        loss_acc = 0.0
        for _ in range(ACCUM):
            _kind, _t1, _t2, _t3 = next(it)
            if _kind == 'lm':
                x = _t1.to(DEV)
                with torch.no_grad():
                    set_graft_mode(model, 'teacher')
                    t_logits = model(x).logits
                set_graft_mode(model, 'student')
                out = model(x, labels=x)   # HF ic kaydirma (dogru next-token)
            else:                          # [RUN5] mesafeli cross-chunk recall batch
                xa, xg, xb = _t1.to(DEV), (_t2.to(DEV) if _t2 is not None else None), _t3.to(DEV)
                _parts = [xa] + ([xg] if xg is not None else []) + [xb]
                with torch.no_grad():
                    set_graft_mode(model, 'teacher')
                    # logits_to_keep: yalniz B pozisyonlarinin logitleri (bellek sinirli)
                    t_logits = model(torch.cat(_parts, 1), logits_to_keep=xb.size(1)).logits
                set_graft_mode(model, 'student')
                model.gradient_checkpointing_disable()   # GC+stream recompute uyusmaz
                enable_streaming(model, True); reset_streaming(model)
                with torch.no_grad():
                    model(xa)              # A: parolayi state'e YAZ (grad yok)
                    if xg is not None:     # ara chunklar: state tasi (grad yok)
                        for _s0 in range(0, xg.size(1), SEQ):
                            model(xg[:, _s0:_s0+SEQ])
                out = model(xb, labels=xb) # B: yalniz bellekten OKU (loss burada)
                enable_streaming(model, False)
                model.gradient_checkpointing_enable()
                x = xb                     # asagidaki del icin
            kl = F.kl_div(F.log_softmax(out.logits/TEMP, -1),
                          F.log_softmax(t_logits/TEMP, -1),
                          log_target=True, reduction='batchmean') * TEMP * TEMP
            loss = out.loss + KL_W * kl
            assert torch.isfinite(loss), f'NaN/Inf @ S2 step {step}'
            (loss / ACCUM).backward()
            loss_acc += loss.item() / ACCUM
        torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        # [OOM-FIX] ara tensorleri hemen birak; parcalanmayi onle
        _lm = float(out.loss.item())   # log icin sakla, sonra birak
        del x, out, t_logits
        if step == 1:
            print(f'[bellek] step1 tepe: {torch.cuda.max_memory_allocated()/1e9:.2f} GB / 14.56 GB', flush=True)
        if step % 25 == 0:
            torch.cuda.empty_cache()
        if step % 50 == 0 or step <= 3:
            print(f'S2 {step}/{STEPS2}  loss {loss_acc:.4f}  lm {_lm:.4f}  {(time.time()-t0)/60:.0f}dk', flush=True)
        if step % 250 == 0:
            save_hfp(f'stage2_{step}')              # [SIGORTA]
    save_hfp('final')
    print('Stage 2 tamam; HFP parametreleri kaydedildi.')
    S2_DONE = True


In [ ]:
# --- 7. VALIDASYON: PPL + NEEDLE + VRAM ---
import math, random, torch

if not globals().get('S2_DONE', False):
    print('S2 bu oturumda kosmadi -> validasyon atlandi (S2 oturumunda kosar).')
else:

    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    # 6a. Kisa-baglam PPL kaybi (kriter: <= 1.05x orijinal)
    set_graft_mode(model, 'student')
    PPL_FINAL = ppl(model, val_ids)
    print(f'PPL: orijinal {PPL_BASE:.2f} -> HFP-hibrit {PPL_FINAL:.2f} '
          f'({PPL_FINAL/PPL_BASE:.3f}x; kriter <=1.05x: {"GECTI" if PPL_FINAL <= 1.05*PPL_BASE else "KALDI"})')

    # 6b. Igne-deligi (needle) — O(1) streaming: grafted state tasinir,
    # full-attn katmanlar icin HF cache kullanilir.
    from transformers import DynamicCache

    def needle_test(model, tok, total_tokens=8192, chunk=512, seed=0):
        random.seed(seed)
        # [SIKILASTIRMA] Egitim listesindeki kelimeler (copper/mountain/blue/
        # silent/river/falcon...) KULLANILMAZ -> ezber itirazi kapanir; yalniz
        # baglamdan okunabilir. Kriter de tam-ifade eslesmesine cekildi.
        secret = random.choice(['orange kettle', 'purple ladder', 'crimson garden'])
        needle = f' The secret passphrase is {secret}. '
        filler = ' The weather was ordinary and nothing of note happened that day.'
        fill_ids = tok(filler, add_special_tokens=False).input_ids
        text_ids = []
        while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
        ins = len(text_ids) // 8                      # igne basa yakin -> uzun gap
        needle_ids = tok(needle, add_special_tokens=False).input_ids
        text_ids = text_ids[:ins] + needle_ids + text_ids[ins:total_tokens - len(needle_ids)]
        query_ids = tok(' The secret passphrase is', add_special_tokens=False).input_ids
        ids = torch.tensor(text_ids + query_ids).unsqueeze(0).to(DEV)

        set_graft_mode(model, 'student'); model.eval()
        enable_streaming(model, True); reset_streaming(model)
        cache = DynamicCache()
        with torch.no_grad():
            for s in range(0, ids.size(1), chunk):
                out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
                cache = out.past_key_values
            gen = []
            last = out.logits[:, -1:].argmax(-1)
            for _ in range(6):
                gen.append(last.item())
                out = model(last, past_key_values=cache, use_cache=True)
                cache = out.past_key_values
                last = out.logits[:, -1:].argmax(-1)
        enable_streaming(model, False)
        answer = tok.decode(gen)
        hit = secret in answer          # tam ifade (iki kelime birden)
        print(f'  L={total_tokens}: gizli="{secret}" cevap="{answer.strip()}" -> {"BULDU" if hit else "BULAMADI"}')
        return hit

    print('Needle (igne) testi:')
    for L in [512, 2048, 8192, 16384]:   # 512 = rejim-yakini kontrol
        needle_test(model, tok, total_tokens=L)

    # 6c. VRAM / state sabitligi: grafted katman state boyutu tokenle degismemeli
    attn = next(m for m in model.modules() if isinstance(m, HFPGraftAttention))
    if attn._stream_state is not None:
        M, z = attn._stream_state[0], attn._stream_state[1]
        print(f'Grafted state: M {tuple(M.shape)} + z {tuple(z.shape)} '
              f'= {(M.numel()+z.numel())*4/1e6:.1f} MB — baglam uzunlugundan BAGIMSIZ (O(1))')
    if DEV == 'cuda':
        print(f'Tepe VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
    print('\nBitince: GPU_ROADMAP.md §10 checklist + RESULTS.md guncellenecek.')